# Google ADK Lab Exercise

Solution to the exercise at the end of Week 5 Day 1 (`5_agent_frameworks/1_google_adk_a2a/lab.ipynb`). The exercise has two parts:

1. Seed a different goal on the board, for example a short haiku about Madrid written to `madrid.txt`, and run the worker again. Does it plan sensible steps and pick the right file tools?
2. Add a fourth tool, a plain typed function, to the worker and watch it show up in the trace.

Run the cells top to bottom with the repo's Python 3.12 kernel. The notebook is self-contained: it keeps its own board file and its own `workspace` folder in this directory, so the lab's board and workspace are untouched.

## Setup

`board.py` and `quiet.py` already live in the day 1 folder, so instead of copying them we put that folder on `sys.path` and import them from there. Two details must happen before the imports:

- `BOARD_PATH` points the board at a local `board.sqlite` in this folder, which is what keeps our runs off the lab's board.
- `silence()` is called before importing ADK, to quiet library logging, exactly as in the lab.

In [ ]:
import os
import sys
from pathlib import Path

# This notebook lives three levels below 5_agent_frameworks, so the day 1 folder is here:
DAY1_FOLDER = Path("../../../1_google_adk_a2a").resolve()
sys.path.insert(0, str(DAY1_FOLDER))

os.environ["BOARD_PATH"] = str(Path("board.sqlite").resolve())  # our own board, not the lab's

from quiet import silence
silence()

import board
from dotenv import load_dotenv
from google.adk.agents import LlmAgent
from google.adk.runners import InMemoryRunner

os.environ.setdefault("GOOGLE_GENAI_USE_VERTEXAI", "FALSE")
load_dotenv(override=True)

MODEL = "gemini-3.1-flash-lite"

## The board tools, unchanged from the lab

The three board tools are copied verbatim: `show_todos` reads the board, `plan_steps` breaks a goal into steps, `complete_task` ticks one off. Each is a plain typed function. The docstring becomes the description the model reads and the type hints become the argument schema.

In [ ]:
def show_todos() -> list[dict]:
    """List every todo on the board. A goal has parent_id None; a step has parent_id set to its goal's id."""
    return board.list_todos()

def plan_steps(goal_id: int, steps: list[str]) -> dict:
    """Break a goal into an ordered checklist of steps on the board. Pass the goal's id and a short list of step descriptions."""
    return {"goal_id": goal_id, "step_ids": [board.add_step(goal_id, step) for step in steps]}

def complete_task(task_id: int, result: str) -> dict:
    """Mark a todo (a step or the goal) with this id as done and record a short result summary."""
    board.complete_todo(task_id, result)
    return {"task_id": task_id, "status": "done"}

## The filesystem MCP server

The same reference server as the lab, started over `npx` and scoped to a `workspace` folder inside this directory, so the agent can only touch files in there. `errlog=subprocess.DEVNULL` quiets its startup logging and is also what lets it run from a Jupyter kernel on Windows.

In [ ]:
import subprocess
from google.adk.tools.mcp_tool import McpToolset, StdioConnectionParams
from mcp import StdioServerParameters

workspace = Path("workspace").resolve()   # the only folder the agent may touch
workspace.mkdir(exist_ok=True)

filesystem = McpToolset(
    connection_params=StdioConnectionParams(
        server_params=StdioServerParameters(
            command="npx",
            args=["-y", "@modelcontextprotocol/server-filesystem", str(workspace)],
            cwd=str(workspace),  # start the server in the workspace so relative file names resolve there
        ),
        timeout=60
    ),
    errlog=subprocess.DEVNULL,
)

## Task 1: a different goal

Seed the haiku goal and let the worker run. The worker and its instruction are the same as the lab's; only the goal on the board is new.

Things to watch in the trace: the worker should read the board, plan a couple of sensible steps under the goal, pick `write_file` to create `madrid.txt`, tick the steps off, and close the goal. Nobody tells it which file tool to use; it chooses from the MCP server's tool descriptions.

In [ ]:
INSTRUCTIONS = """
You are a careful worker with a shared todo board and a set of file tools.

Take the pending goal and see it through. Begin by laying out a short plan: the handful of concrete steps the work itself breaks down into, added to the board under the goal. Then carry them out with your file tools, marking each step done as you finish it. Once the steps are all done, close the goal. Your files live in the single folder your tools are allowed to use.
"""

worker = LlmAgent(
    model=MODEL,
    name="task_worker",
    instruction=INSTRUCTIONS,
    tools=[show_todos, plan_steps, complete_task, filesystem],
)

board.reset_board()
goal_id = board.add_goal("Write a short haiku about Madrid into madrid.txt.")
board.claim_todo(goal_id)

result = await InMemoryRunner(agent=worker).run_debug("Please work the pending goal on the board.", verbose=True)

Now check the outcome: the board should show the goal and its steps struck through, and `madrid.txt` should hold the haiku.

In [ ]:
board.show_board()
print("\nmadrid.txt:\n" + (workspace / "madrid.txt").read_text(encoding="utf-8"))

## Task 2: a fourth tool

The fourth tool is `count_words`, one more plain typed function with a docstring. Nothing else is needed to register it: it just joins the same `tools` list.

To make sure the tool actually shows up in the trace, the new goal asks for the word count to be recorded on the board. The model cannot count words reliably on its own, so it has to call the tool.

In [ ]:
def count_words(text: str) -> dict:
    """Count the words in a piece of text and return the word count."""
    return {"word_count": len(text.split())}

worker_with_counter = LlmAgent(
    model=MODEL,
    name="task_worker",
    instruction=INSTRUCTIONS,
    tools=[show_todos, plan_steps, complete_task, count_words, filesystem],
)

board.reset_board()
goal_id = board.add_goal(
    "Write a short haiku about Barcelona into barcelona.txt, "
    "count the words in the haiku with the count_words tool, "
    "and record the word count when you close the goal."
)
board.claim_todo(goal_id)

result = await InMemoryRunner(agent=worker_with_counter).run_debug("Please work the pending goal on the board.", verbose=True)

In [ ]:
board.show_board()
print("\nbarcelona.txt:\n" + (workspace / "barcelona.txt").read_text(encoding="utf-8"))